# Tahap 09: Cross-Domain Validation (CDV)

Tahap ini bertujuan untuk menguji kemampuan generalisasi model pada data yang berasal dari sumber atau domain yang berbeda (*Cross-Domain Validation*). Proses diawali dengan mengidentifikasi seluruh variasi domain asal data (kolom `source_file`) yang terdapat pada dataset sebelum dilakukan pemisahan (*splitting*).

In [ ]:
# --- LIBRARIES ---
import os
import pandas as pd

# --- 1. KONFIGURASI DIREKTORI & PATH ---
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'

# Memuat data prapemrosesan (sebelum splitting) yang masih memiliki informasi 'source_file'
df_pre = pd.read_csv(os.path.join(BASE_DIR, 'data_preprocessing_2.0.csv'))

# --- 2. IDENTIFIKASI DOMAIN DATA ---
print("="*60)
print("DAFTAR DOMAIN DATA (SOURCE FILE)")
print("="*60)

domains = df_pre['source_file'].unique()
for i, domain in enumerate(domains, 1):
    print(f"{i}. {domain}")

print("-" * 60)
print(f"Total: {len(domains)} Domain teridentifikasi.")
print("="*60)

## 9.1 Eksekusi Validasi Silang Antar Domain
Pengujian dilakukan menggunakan metode `StratifiedGroupKFold` dengan 5 *splits*. Metode ini memastikan bahwa distribusi label (*Pujian*, *Keluhan*, *Saran*) tetap terjaga proporsinya pada setiap lipatan (*fold*), sekaligus menjamin bahwa data dari satu grup domain yang sama tidak terpisah antara set pelatihan dan set pengujian.

In [ ]:
# --- LIBRARIES TAMBAHAN ---
import pickle
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# --- 1. PEMUATAN DATA FITUR CHI-SQUARE ---
print("[1/3] Memuat data fitur Chi-Square dari arsip...")
path_chi = os.path.join(BASE_DIR, 'selector_chi2_rev_2.0.pkl')

with open(path_chi, 'rb') as f:
    # Memuat fitur dan label yang telah diekstraksi
    X_train_chi, X_test_chi, y_train_raw_chi, y_test_raw_chi = pickle.load(f)

# --- 2. SINKRONISASI INFORMASI DOMAIN (GROUPS) ---
print("[2/3] Melakukan sinkronisasi informasi domain dengan data pelatihan...")

df_train_final = pd.read_csv(os.path.join(BASE_DIR, 'data_train_final_2.0.csv'))

# Membuat kamus pemetaan (mapping) dari teks komentar ke source_file
mapping_dict = df_pre.drop_duplicates('text').set_index('text')['source_file'].to_dict()

# Menggabungkan source_file ke dalam data training berdasarkan teks
groups = df_train_final['text'].map(mapping_dict)

# Penanganan mitigasi untuk data yang gagal terpetakan (Missing Values)
if groups.isnull().any():
    groups = groups.fillna('unknown')
groups = groups.values

# Konversi Label Target ke format numerik diskrit
custom_mapping = {'keluhan': 0, 'saran': 1, 'pujian': 2}
y_cv = df_train_final['label_pks'].map(custom_mapping).values

print(f"      Status: Variabel X_train_chi dan Groups siap ({len(groups)} baris, {len(np.unique(groups))} domain unik).")

# --- 3. KONFIGURASI MODEL & CROSS-VALIDATION ---
# Inisialisasi pembagian K-Fold (5 Lipatan)
sgkf = StratifiedGroupKFold(n_splits=5)

# Inisialisasi model Logistic Regression dengan konfigurasi utama
model_cv = LogisticRegression(
    multi_class='multinomial', 
    solver='lbfgs', 
    max_iter=1000, 
    class_weight='balanced', 
    random_state=42
)

print("[3/3] Menjalankan 5-Fold Cross-Validation antar Domain...")
# Proses komputasi validasi silang
scores = cross_val_score(model_cv, X_train_chi, y_cv, cv=sgkf, groups=groups)

# --- 4. LAPORAN HASIL EVALUASI CDV ---
print("\n" + "="*60)
print("HASIL VALIDASI ANTAR DOMAIN (FINAL)")
print("="*60)

for i, score in enumerate(scores, 1):
    print(f"Iterasi {i} : Akurasi {score:.2%}")

print("-" * 60)
mean_acc = np.mean(scores)
std_acc = np.std(scores)

print(f"RATA-RATA AKURASI TOTAL : {mean_acc:.2%}")
print(f"STANDAR DEVIASI (STD)   : {std_acc:.4f}")
print("="*60)